# IEEE39 rollback/Newton validation: native, FMI ME, and FMI CS

This notebook compares fresh, closed-loop DPsim runs of the same IEEE39 EMT network. The averaged inverter is at bus 30. All methods start from the solved powerflow and receive the same 100 MW bus-39 load step at 2.5 s. The tested communication steps are 100, 500, and 1000 µs.

The `cs` FMU now supports FMI state checkpoint/replay and uses the opt-in electrical rollback/Newton path with four midpoint voltage replay substeps. Its exact exponential current update remains the FMU's own integration method; native DPsim and ME use trapezoidal integration. Thus the comparison is closed-loop implementation agreement with distinct CS integration, not a claim of identical discretization or independent physical accuracy.

Regenerate the runs from the repository root with:

```bash
python3 -u dpsim-fmi/examples/ieee39_inverter/run_ieee39_dpsim_native_me_cs_sweep.py \
  --steps-us 100 500 1000 --duration-s 5 --load-step-s 2.5 \
  --output-name ieee39-rollback-5s-sweep
```


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

root = Path.cwd().resolve()
while root != root.parent and not (root / "dpsim-fmi").is_dir():
    root = root.parent
if not (root / "dpsim-fmi").is_dir():
    raise FileNotFoundError("Open this notebook from a DPsim checkout")
results_dir = root / "build-fmi" / "ieee39-inverter" / "ieee39-rollback-5s-sweep"
metrics_path = results_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(f"Results: {metrics_path}")
print(metrics["configuration"])

## Full-run errors and timing

In [ ]:
rows = []
for step, case in metrics["cases"].items():
    for method in ("native", "me", "cs"):
        r = case["runtimes"][method]
        comp = case["comparisons"].get(method)
        s = comp["full_run"] if comp else {}
        ni = r.get("newton_metrics", {})
        rows.append(
            {
                "timestep [µs]": int(step),
                "method": method.upper(),
                "PCC voltage RMS error [%]": s.get("voltage_rms_error_percent"),
                "inverter current RMS error [%]": s.get("current_rms_error_percent"),
                "Newton max iterations": ni.get("max_iterations"),
                "max scaled residual": ni.get("max_scaled_residual"),
                "solver wall [s]": r["solver_wall_s"],
                "process wall [s]": r["process_wall_s"],
            }
        )
summary = pd.DataFrame(rows).sort_values(["timestep [µs]", "method"])
display(
    summary.style.format(
        {c: "{:.5g}" for c in summary.select_dtypes("number").columns}, na_rep="—"
    )
)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for method, group in summary.groupby("method"):
    if method != "NATIVE":
        ax[0].plot(
            group["timestep [µs]"],
            group["PCC voltage RMS error [%]"],
            "o-",
            label=method,
        )
        ax[1].plot(
            group["timestep [µs]"],
            group["inverter current RMS error [%]"],
            "o-",
            label=method,
        )
    ax[2].plot(group["timestep [µs]"], group["solver wall [s]"], "o-", label=method)
for a, title, ylabel in zip(
    ax,
    [
        "PCC voltage error vs native DPsim",
        "Inverter current error vs native DPsim",
        "Solver wall time",
    ],
    [
        "Full-run three-phase RMS error [%]",
        "Full-run three-phase RMS error [%]",
        "Run time [s]",
    ],
):
    a.set_xscale("log")
    a.set_xlabel("Fixed step [µs]")
    a.set_ylabel(ylabel)
    a.set_title(title)
    a.grid(True, alpha=0.3)
    a.legend()
plt.tight_layout()

## Initialization check

In [ ]:
init_rows = []
for step, case in metrics["cases"].items():
    for method, comp in case["comparisons"].items():
        x = comp["initial"]
        init_rows.append(
            {
                "step [µs]": int(step),
                "method": method.upper(),
                "native Vabc at t=0 [V]": x["native_v_abc_V"],
                "candidate Vabc at t=0 [V]": x["candidate_v_abc_V"],
                "native Iabc at t=0 [A]": x["native_i_abc_A"],
                "candidate Iabc at t=0 [A]": x["candidate_i_abc_A"],
            }
        )
display(pd.DataFrame(init_rows))

## Per-timestep error traces and load-step response

For every timestep, the first figure shows the trailing 100 ms three-phase RMS error. The second shows cumulative RMS error from time zero through each point, estimated from the retained compact traces; the full-run summary above uses full-resolution data. The lower waveform figure zooms to ±25 ms around the 2.5 s event and overlays phase-a PCC voltage and inverter current for all three methods.

In [ ]:
for step, case in sorted(metrics["cases"].items(), key=lambda item: int(item[0])):
    step = int(step)
    print(f"### {step} µs")
    me = case["comparisons"]["me"]
    cs = case["comparisons"]["cs"]
    em = pd.read_csv(me["rolling_error_trace"])
    ec = pd.read_csv(cs["rolling_error_trace"])
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for frame, label in [(em, "DPsim ME"), (ec, "DPsim CS")]:
        ax[0].plot(frame.time_s, frame.pcc_voltage_rms_error_percent_100ms, label=label)
        ax[1].plot(
            frame.time_s, frame.inverter_current_rms_error_percent_100ms, label=label
        )
    ax[0].set_ylabel("PCC voltage RMS error [%]")
    ax[1].set_ylabel("Inverter current RMS error [%]")
    ax[1].set_xlabel("Time [s]")
    for a in ax:
        a.axvline(2.5, color="0.3", ls=":", label="100 MW load step")
        a.grid(True, alpha=0.3)
        a.legend()
    fig.suptitle(f"{step} µs: rolling 100 ms error vs native DPsim")
    plt.tight_layout()
    plt.show()

    native_path = case["runtimes"]["native"]["directory"] + "/native_trace.csv"
    native = pd.read_csv(native_path)
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for mode, label in (("me", "DPsim ME"), ("cs", "DPsim CS")):
        candidate = pd.read_csv(case["comparisons"][mode]["compact_trace"])
        if len(candidate) != len(native) or not np.allclose(
            candidate.time_s, native.time_s, rtol=0, atol=1e-12
        ):
            raise ValueError(
                f"{step} µs {label} compact trace does not align with native DPsim"
            )
        vcols = ["pcc_v_a_V", "pcc_v_b_V", "pcc_v_c_V"]
        icols = ["current_a_A", "current_b_A", "current_c_A"]
        vdiff = (candidate[vcols].to_numpy() - native[vcols].to_numpy()) ** 2
        idiff = (candidate[icols].to_numpy() - native[icols].to_numpy()) ** 2
        vref = native[vcols].to_numpy() ** 2
        iref = native[icols].to_numpy() ** 2
        verror = 100 * np.sqrt(
            np.cumsum(vdiff.sum(axis=1)) / np.cumsum(vref.sum(axis=1))
        )
        ierror = 100 * np.sqrt(
            np.cumsum(idiff.sum(axis=1)) / np.cumsum(iref.sum(axis=1))
        )
        ax[0].plot(native.time_s, verror, label=label)
        ax[1].plot(native.time_s, ierror, label=label)
    ax[0].set_ylabel("Cumulative PCC voltage RMS error [%]")
    ax[1].set_ylabel("Cumulative inverter current RMS error [%]")
    ax[1].set_xlabel("Time [s]")
    for a in ax:
        a.axvline(2.5, color="0.3", ls=":", label="100 MW load step")
        a.grid(True, alpha=0.3)
        a.legend()
    fig.suptitle(f"{step} µs: cumulative RMS error since t = 0")
    plt.tight_layout()
    plt.show()

    zm = pd.read_csv(me["load_step_zoom_trace"])
    zc = pd.read_csv(cs["load_step_zoom_trace"])
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax[0].plot(zm.time_s, zm.native_v_a_V, "k--", label="Native DPsim")
    ax[0].plot(zm.time_s, zm.me_v_a_V, label="DPsim ME")
    ax[0].plot(zc.time_s, zc.cs_v_a_V, label="DPsim CS")
    ax[1].plot(zm.time_s, zm.native_i_a_A, "k--", label="Native DPsim")
    ax[1].plot(zm.time_s, zm.me_i_a_A, label="DPsim ME")
    ax[1].plot(zc.time_s, zc.cs_i_a_A, label="DPsim CS")
    ax[0].set_ylabel("Bus 30 phase-a voltage [V]")
    ax[1].set_ylabel("Inverter phase-a current [A]")
    ax[1].set_xlabel("Time [s]")
    for a in ax:
        a.grid(True, alpha=0.3)
        a.legend()
    fig.suptitle(f"{step} µs: 2.5 s load-step zoom")
    plt.tight_layout()
    plt.show()

## Interpretation notes

At 100 µs, CS matches native to 0.0000365% voltage RMS and 0.0223% current RMS over five seconds. At 500 µs the errors are 0.00091% and 0.558%; at 1000 µs they are 0.00361% and 2.26%. CS converges in two Newton iterations in these cases. The separate 5 ms stress case has 64.2% current RMS error, so it is too coarse for this 60 Hz network and should not be presented as a normal operating timestep. See the example README for the 10 s / 100 µs validation.
